# Top Search Queries (Sitewide)

Use this notebook to see which Google search queries drive the most traffic to `lifeline.org.au` overall (not split by page).

### Quick start (beginner-friendly)
1. Run the **Setup (run once)** cell.
2. In **Parameters**, choose `DAYS_BACK` and `TOP_N`.
3. Run the remaining cells from top to bottom.

### Links
- GitHub repo: [github.com/aidanm-lla/lla-data](https://github.com/aidanm-lla/lla-data)
- Open this notebook in Colab: [Top Search Queries (Sitewide)](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/07_top_search_queries_sitewide.ipynb)

### Other notebooks
- [Search Contribution Overview](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/01_search_contribution_overview.ipynb)
- [Top Pages Search Performance](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/02_top_pages_search_performance.ipynb)
- [Query Drivers by Page](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/03_query_drivers_by_page.ipynb)
- [Opportunity Watchlist](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/04_opportunity_watchlist.ipynb)
- [Traffic Source Quality](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/05_traffic_sources.ipynb)
- [Time Patterns for Crisis-Related Pages](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/06_time_patterns.ipynb)


In [ ]:
#@title Setup (run once)
import sys
import os

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
    if not os.path.exists("lla-data"):
        !git clone -q https://github.com/aidoanto/lla-data.git
    repo = os.path.abspath("lla-data")
    if repo not in sys.path:
        sys.path.insert(0, repo)
    !pip install -U -q "plotly>=6.1.1" "kaleido>=1.2.0"
else:
    for p in ("..", "../.."):
        ap = os.path.abspath(p)
        if ap not in sys.path:
            sys.path.insert(0, ap)

import plotly.express as px

import lifeline_theme
from lla_data import config
from lla_data.bq import build_date_params, default_query_window, get_client, load_sql_template, run_query

lifeline_theme.inject_fonts()

client = get_client()


In [ ]:
#@title Parameters
DAYS_BACK = 28 #@param {type:"integer"}
TOP_N = 25 #@param {type:"integer"}

window = default_query_window(DAYS_BACK)


In [ ]:
from google.cloud import bigquery

query = load_sql_template(
    "sql/notebooks/seo_top_queries_sitewide.sql",
    project_id=config.PROJECT_ID,
    searchconsole_dataset=config.SEARCHCONSOLE_DATASET,
)

params = build_date_params(window) + [
    bigquery.ScalarQueryParameter("top_n", "INT64", TOP_N),
]

df_queries = run_query(client, query, params=params)
df_queries.head(20)


In [ ]:
analysis_window_label = f"{window.start_date} to {window.end_date}"

df_plot = df_queries.copy()
total_clicks = float(df_plot["clicks"].sum()) or 1.0
df_plot["click_share"] = df_plot["clicks"] / total_clicks
df_plot["query_label"] = df_plot["query"].astype(str).str.slice(0, 80)

fig = px.bar(
    df_plot.sort_values("clicks", ascending=True),
    x="clicks",
    y="query_label",
    orientation="h",
    hover_name="query",
    hover_data={"query_label": False, "impressions": True, "ctr": ":.1%", "avg_position": ":.2f", "click_share": ":.1%"},
    template="lifeline",
    title=f"Top {TOP_N} Search Queries Driving Traffic ({analysis_window_label})",
)
fig.update_layout(height=700, margin={"l": 350, "r": 40, "t": 80, "b": 40})
fig.update_yaxes(automargin=True)
lifeline_theme.add_lifeline_logo(fig)
fig.show()

df_output = df_plot[["query", "clicks", "impressions", "ctr", "avg_position", "click_share"]].copy()
df_output["ctr"] = df_output["ctr"].round(4)
df_output["avg_position"] = df_output["avg_position"].round(2)
df_output["click_share"] = df_output["click_share"].round(4)
df_output
